# Part 7 · Notebook 03 — Mean reversion and range-bound strategies

**Sessions:** S3 (Mean reversion & range-bound groups) · [Lesson plan](../../docs/lessons/PART_07_STRATEGY_LIBRARY.md) · graded labs in [`labs/part07/`](../../labs/part07/)

**You will:**
1. Compute internal bar strength with the flat-bar guard.
2. Give a z-score reversion trade a time stop.
3. Put a trend filter on a short-term RSI(2) entry.
4. See mean reversion earn in ranges and lose in trends.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic, built from regimes you know, and every strategy here is a **hypothesis** with a first-look evaluation: the honest backtest comes in Part 8.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p7lib.py is in notebooks/part07/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p7lib as p

p.use_course_style()

In [ ]:
bars = p.regime_market()
o, h, l, c = (bars[k].to_numpy() for k in ("open", "high", "low", "close"))

## 1. Internal bar strength

`IBS = (C − L) / (H − L)`: 0 when the bar closed on its low, 1 on its high. A low IBS is a classic short-term buy signal in index ETFs. On a flat bar (H = L) return **0.5**, not a division by zero.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def ibs(high, low, close):
    rng_ = high - low
    return ...                                    # ✍️ np.divide(..., out=np.full(close.shape, 0.5), where=rng_ > 0)

h2, l2, c2 = np.append(h, 50.0), np.append(l, 50.0), np.append(c, 50.0)       # one flat bar at the end
mine = p.attempt(ibs, h2, l2, c2)
mine = p.check("ibs", mine, p.ibs(h2, l2, c2))
np.round(mine[-4:], 3)

## 2. Z-score reversion with a time stop

`z = (C − SMA(n)) / rolling std`. Flat → long when `z < −entry`, short when `z > entry`. In a trade → exit when the price has come back (`|z| < exit_`) **or** after `max_hold` bars. The time stop is the admission that a move that doesn't revert is a trend. `held` counts the bars since entry (1 on the entry bar). Write the in-trade branch.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def zscore_reversion(close, n=20, entry=2.0, exit_=0.5, max_hold=10):
    z = p.rolling_z(close, n)
    pos, held = np.zeros(z.size), 0
    for t in range(1, z.size):
        if not np.isfinite(z[t]):
            continue
        if pos[t - 1] == 0:
            pos[t] = 1.0 if z[t] < -entry else (-1.0 if z[t] > entry else 0.0)
            held = 1 if pos[t] else 0
        else:
            held += 1
            pos[t] = ...                          # ✍️ 0 if reverted or held too long, else keep pos[t − 1]
    return pos

mine = p.attempt(zscore_reversion, c)
mine = p.check("zscore_reversion", mine, p.zscore_reversion(c))
print(f"in a trade {np.mean(np.asarray(mine) != 0):.0%} of bars")

## 3. RSI(2) with a trend filter

Connors' rule: buy a sharp short-term dip (`RSI(2) < entry`), but **only while the long-term trend is up** (`close > SMA(trend_n)`), and sell when `RSI(2) > exit_`. Write the entry condition.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def rsi2(close, entry=10, exit_=70, trend_n=200, use_trend=True):
    r, m = p.rsi(close, 2), p.sma(close, trend_n)
    pos = np.zeros(close.size)
    for t in range(1, close.size):
        if pos[t - 1] == 0:
            ok = ...                              # ✍️ RSI(2) below entry, and (no filter, or close above its SMA)
            pos[t] = 1.0 if ok else 0.0
        else:
            pos[t] = 0.0 if r[t] > exit_ else 1.0
    return pos

mine = [p.attempt(rsi2, c, use_trend=True), p.attempt(rsi2, c, use_trend=False)]
mine = p.check("rsi2", mine, [p.rsi2_reversion(c, use_trend=True), p.rsi2_reversion(c, use_trend=False)])
print(f"trades taken: {int(np.sum(np.diff(mine[0]) > 0))} with the filter, {int(np.sum(np.diff(mine[1]) > 0))} without")

## 4. Where mean reversion works

Same regime table as for momentum. The time stop and the trend filter don't make mean reversion work in a trend; they limit how much it loses there.

In [ ]:
signals = {"z-score (time stop 10)": p.zscore_reversion(c), "z-score (no time stop)": p.zscore_reversion(c, max_hold=10_000),
           "RSI(2) + trend filter": p.rsi2_reversion(c), "RSI(2), no filter": p.rsi2_reversion(c, use_trend=False),
           "IBS < 0.2, one day": (p.ibs(h, l, c) < 0.2).astype(float)}
display(p.regime_table(bars, signals).round(2))
worst = {k: pd.Series(p.quick_eval(v, o)["pnl"]).rolling(20).sum().min() for k, v in signals.items()}
print("worst 20-day P&L:", {k: f"{v:.1%}" for k, v in worst.items()})

## Wrap-up

* Mean reversion is the mirror image of momentum: it earns in ranges and pays in trends.
* Always add a regime filter and a maximum holding period, and state them in the spec.
* Graded version: `labs/part07/week23_linear_groups` (RSI(2), IBS, z-score, Bollinger fade gated by ADX, opening-range breakout).